# مشروع Smart Leaf Doctor: اكتشاف أمراض أوراق النباتات 🌿

**الهدف من المشروع:**
بناء نظام ذكي يعتمد على الشبكات العصبية التلافيفية (CNN) ونقل التعلم (Transfer Learning) لاكتشاف جميع الأمراض والفئات (38 فئة) المتوفرة في مجموعة البيانات.

**لماذا نستخدم نقل التعلم (Transfer Learning)؟**
نقل التعلم يعني استخدام نموذج مدرب مسبقاً (MobileNetV2). هذا النموذج قوي جداً في استخراج الخصائص مما يمكننا من تصنيف عدد كبير من الفئات (38 فئة) بدقة وكفاءة عالية.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

2026-05-01 14:14:24.972737: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. تحميل وتجهيز مجموعة البيانات (Dataset Loading)

سنحاول أولاً استخدام `kaggle` لتحميل البيانات، وفي حالة نجاحه أو وجود البيانات محلياً، سنستخدم جميع الفئات (المجلدات) المتوفرة للتدريب والاختبار.

In [2]:
LOCAL_DATASET_DIR = '../dataset/'
BASE_DIR = None

try:
    import kagglehub
    if not os.path.exists(os.path.join(LOCAL_DATASET_DIR, 'New Plant Diseases Dataset(Augmented)')):
        print("جاري تحميل مجموعة البيانات من كاجل...")
        path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
        print("تم التحميل بنجاح في:", path)
        BASE_DIR = os.path.join(path, "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)")
    else:
        BASE_DIR = os.path.join(LOCAL_DATASET_DIR, 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)')
        print("البيانات موجودة محلياً في:", BASE_DIR)
except Exception as e:
    print(f"حدث خطأ أثناء محاولة تحميل البيانات: {e}")
    print("الرجاء وضع البيانات يدوياً في مجلد dataset/")
    BASE_DIR = os.path.join(LOCAL_DATASET_DIR, 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)')

جاري تحميل مجموعة البيانات من كاجل...
تم التحميل بنجاح في: /Users/abdallahzarea/.cache/kagglehub/datasets/vipoooool/new-plant-diseases-dataset/versions/2


In [3]:
if BASE_DIR and os.path.exists(BASE_DIR):
    train_dir = os.path.join(BASE_DIR, 'train')
    valid_dir = os.path.join(BASE_DIR, 'valid')
    
    print("تم إعداد مسارات التدريب والاختبار لجميع الفئات.")
    
    # تأكيد عدد الصور
    for split_dir, split_name in [(train_dir, "Training"), (valid_dir, "Validation")]:
        print(f"\n--- {split_name} Set ---")
        total = 0
        if os.path.exists(split_dir):
            classes = os.listdir(split_dir)
            print(f"Number of classes: {len(classes)}")
            for c in classes:
                class_path = os.path.join(split_dir, c)
                if os.path.isdir(class_path):
                    count = len(os.listdir(class_path))
                    total += count
            print(f"Total: {total} images")
else:
    print("الرجاء التأكد من مسار البيانات. قد تحتاج إلى تحميل البيانات يدوياً.")

تم إعداد مسارات التدريب والاختبار لجميع الفئات.

--- Training Set ---
Number of classes: 38
Total: 70295 images

--- Validation Set ---
Number of classes: 38
Total: 17572 images


## 2. المعالجة المسبقة وتكبير البيانات (Preprocessing & Data Augmentation)

In [4]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64 # تم زيادة حجم الدفعة نظراً لكبر حجم البيانات لتسريع التدريب

if os.path.exists(train_dir):
    train_datagen = ImageDataGenerator(
        rescale=1./255,          
        rotation_range=20,       
        zoom_range=0.2,          
        horizontal_flip=True,    
        brightness_range=[0.8, 1.2]
    )
    
    valid_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    
    valid_generator = valid_datagen.flow_from_directory(
        valid_dir,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    class_names = list(train_generator.class_indices.keys())
    num_classes = len(class_names)
    print(f"تم العثور على {num_classes} فئة.")
    
    import json
    os.makedirs('../models', exist_ok=True)
    with open('../models/class_names.json', 'w') as f:
        json.dump(class_names, f)
    print("تم حفظ أسماء الفئات في models/class_names.json")

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
تم العثور على 38 فئة.
تم حفظ أسماء الفئات في models/class_names.json


## 3. نموذج MobileNetV2 (Transfer Learning)

نظراً لضخامة البيانات (38 فئة)، سنتخطى النموذج البسيط ونستخدم MobileNetV2 مباشرة للحصول على أفضل دقة وتوفير الوقت.

In [5]:
if os.path.exists(train_dir) and 'num_classes' in locals():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False # تجميد الأوزان الأساسية
    
    mobilenet_model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        Dropout(0.4),
        Dense(num_classes, activation='softmax') # عدد ديناميكي يمثل جميع الفئات
    ])
    
    mobilenet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    mobilenet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │        19,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,933,350 (11.19 MB)

 Trainable params: 675,366 (2.58 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
if os.path.exists(train_dir) and 'mobilenet_model' in locals():
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ModelCheckpoint('../models/smart_leaf_doctor_mobilenetv2.h5', monitor='val_accuracy', save_best_only=True)
    ]
    
    epochs = 10 
    history_mobilenet = mobilenet_model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=epochs,
        callbacks=callbacks
    )

Epoch 1/10
  17/1099 ━━━━━━━━━━━━━━━━━━━━ 36:49 2s/step - accuracy: 0.1359 - loss: 3.6558

KeyboardInterrupt: 

## 4. تقييم النموذج (Model Evaluation)

In [ ]:
if os.path.exists(train_dir) and 'history_mobilenet' in locals():
    os.makedirs('../outputs', exist_ok=True)
    
    # منحنى الدقة (Accuracy Curve)
    plt.figure(figsize=(8, 6))
    plt.plot(history_mobilenet.history['accuracy'], label='Training Accuracy')
    plt.plot(history_mobilenet.history['val_accuracy'], label='Validation Accuracy')
    plt.title('منحنى الدقة (Model Accuracy)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('../outputs/accuracy_curve.png')
    plt.show()
    
    # منحنى الخطأ (Loss Curve)
    plt.figure(figsize=(8, 6))
    plt.plot(history_mobilenet.history['loss'], label='Training Loss')
    plt.plot(history_mobilenet.history['val_loss'], label='Validation Loss')
    plt.title('منحنى الخطأ (Model Loss)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('../outputs/loss_curve.png')
    plt.show()

In [ ]:
if os.path.exists(train_dir) and 'history_mobilenet' in locals():
    # الحصول على التنبؤات
    valid_generator.reset()
    predictions = mobilenet_model.predict(valid_generator)
    y_pred = np.argmax(predictions, axis=1)
    y_true = valid_generator.classes
    
    # مصفوفة الارتباك (سيتم حفظها كصورة نظراً لضخامتها)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(20, 18))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.title('مصفوفة الارتباك (Confusion Matrix)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig('../outputs/confusion_matrix.png')
    plt.show()

## 5. دالة التنبؤ وتجربتها

In [ ]:
from tensorflow.keras.preprocessing import image

if os.path.exists(train_dir) and 'mobilenet_model' in locals():
    def predict_leaf_disease(img_path):
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) / 255.0
        
        preds = mobilenet_model.predict(img_array)[0]
        pred_idx = np.argmax(preds)
        predicted_class = class_names[pred_idx]
        confidence = preds[pred_idx] * 100
        
        status = "Healthy" if "healthy" in predicted_class.lower() else "Diseased"
        return predicted_class, confidence, status

    # عرض عينة (Sample Prediction)
    sample_class = random.choice(class_names)
    sample_dir = os.path.join(valid_dir, sample_class)
    if os.path.exists(sample_dir) and len(os.listdir(sample_dir)) > 0:
        sample_img = random.choice(os.listdir(sample_dir))
        sample_path = os.path.join(sample_dir, sample_img)
        
        predicted_class, confidence, status = predict_leaf_disease(sample_path)
        
        img = cv2.imread(sample_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img)
        plt.title(f"True: {sample_class}\nPred: {predicted_class}\nConf: {confidence:.1f}%")
        plt.axis('off')
        plt.savefig('../outputs/sample_predictions.png')
        plt.show()